# Hands-on 29 (Bab 11): Mini Proyek Klasifikasi Sederhana di Raspberry Pi / Emulator

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Mini proyek end-to-end meniru deployment edge: simulator sensor -> model lokal
(joblib dari Hands-on 27) -> status klasifikasi -> log CSV dan/atau MQTT.

## Tujuan

Menyusun mini proyek end-to-end yang meniru deployment nyata di Raspberry Pi atau emulator Linux sejenis. Tujuannya:

- membaca data sensor atau simulasi sensor,
- menjalankan model lokal,
- menghasilkan status klasifikasi,
- dan, bila diinginkan, meneruskan hasil ke MQTT atau log file.


In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")


[INFO] Working directory: /downloads/iot-ml-book


## 1. Mini Proyek tanpa MQTT (simulator -> model -> log CSV)

In [2]:
print(jalankan_buku("scripts/edge_room_classifier_project.py", 90))

=== Mini proyek edge room classifier dimulai ===
step=0 | T=32.18 H=54.73 CO2=620 P=149.0 -> status=warning
step=1 | T=26.33 H=44.3 CO2=1482 P=279.15 -> status=fault
step=2 | T=26.11 H=68.75 CO2=560 P=126.81 -> status=normal
step=3 | T=30.71 H=49.79 CO2=474 P=253.97 -> status=warning
step=4 | T=25.3 H=52.49 CO2=742 P=187.19 -> status=normal
step=5 | T=32.0 H=56.79 CO2=1083 P=234.89 -> status=warning
step=6 | T=26.23 H=47.58 CO2=1042 P=142.92 -> status=warning
step=7 | T=27.79 H=47.8 CO2=859 P=163.91 -> status=normal
step=8 | T=24.77 H=58.38 CO2=1353 P=184.33 -> status=normal
step=9 | T=25.06 H=69.16 CO2=578 P=205.1 -> status=normal
step=10 | T=28.36 H=40.37 CO2=1167 P=142.49 -> status=warning
step=11 | T=26.1 H=41.79 CO2=1212 P=118.42 -> status=normal
step=12 | T=28.67 H=50.23 CO2=819 P=275.54 -> status=normal
step=13 | T=31.95 H=42.09 CO2=838 P=210.02 -> status=warning
step=14 | T=27.91 H=50.61 CO2=1429 P=240.7 -> status=normal
step=15 | T=30.46 H=44.68 CO2=632 P=128.77 -> status=warn

## 2. Verifikasi Log CSV

In [3]:
import pandas as pd
log = pd.read_csv("outputs/reports/edge_room_predictions.csv")
print(log.head(10).to_string(index=False))
print("\nDistribusi status:", log["predicted_status"].value_counts().to_dict())

 step  temperature_c  humidity_pct  co2_ppm  power_watt predicted_status
    0          32.18         54.73      620      149.00          warning
    1          26.33         44.30     1482      279.15            fault
    2          26.11         68.75      560      126.81           normal
    3          30.71         49.79      474      253.97          warning
    4          25.30         52.49      742      187.19           normal
    5          32.00         56.79     1083      234.89          warning
    6          26.23         47.58     1042      142.92          warning
    7          27.79         47.80      859      163.91           normal
    8          24.77         58.38     1353      184.33           normal
    9          25.06         69.16      578      205.10           normal

Distribusi status: {'normal': 14, 'warning': 13, 'fault': 3}


## 3. Versi dengan MQTT (broker lokal)

In [4]:
import json
import paho.mqtt.subscribe as mqttsub
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    # Subscriber penerima prediksi (skrip buku scripts/mqtt/subscribe_sensor.py memakai topik room01/telemetry,
    # jadi di sini dipakai paho.mqtt.subscribe untuk topik prediksi), lalu skrip edge buku dijalankan apa adanya.
    import threading
    hasil = {}
    def dengar():
        hasil["msgs"] = mqttsub.simple("iotlab/room01/prediction", hostname="127.0.0.1", port=1883, qos=1, msg_count=6)
    t = threading.Thread(target=dengar); t.start()
    time.sleep(1)
    out = jalankan_buku("scripts/edge_room_classifier_mqtt.py", 30)
    print(out)
    ringkas = __import__("re").findall(r"'predicted_status': '(\w+)'", out)
    print("Jumlah publish:", len(ringkas), "| distribusi:", dict(__import__("collections").Counter(ringkas)))
    t.join(timeout=10)
    print("--- Pesan prediksi yang diterima subscriber ---")
    for m in hasil.get("msgs", []):
        print(m.topic, m.payload.decode())
finally:
    broker.terminate()

Published: {'step': 0, 'temperature_c': 30.43, 'humidity_pct': 44.05, 'co2_ppm': 647, 'power_watt': 235.57, 'predicted_status': 'warning'}
Published: {'step': 1, 'temperature_c': 29.43, 'humidity_pct': 57.48, 'co2_ppm': 460, 'power_watt': 235.73, 'predicted_status': 'warning'}
Published: {'step': 2, 'temperature_c': 31.01, 'humidity_pct': 46.11, 'co2_ppm': 813, 'power_watt': 205.84, 'predicted_status': 'warning'}
Published: {'step': 3, 'temperature_c': 31.24, 'humidity_pct': 49.56, 'co2_ppm': 1186, 'power_watt': 117.88, 'predicted_status': 'warning'}
Published: {'step': 4, 'temperature_c': 31.9, 'humidity_pct': 69.82, 'co2_ppm': 837, 'power_watt': 105.64, 'predicted_status': 'warning'}
Published: {'step': 5, 'temperature_c': 32.47, 'humidity_pct': 68.14, 'co2_ppm': 967, 'power_watt': 215.65, 'predicted_status': 'warning'}
Published: {'step': 6, 'temperature_c': 32.86, 'humidity_pct': 54.3, 'co2_ppm': 1251, 'power_watt': 215.37, 'predicted_status': 'fault'}
Published: {'step': 7, 'tempe

## Penutup

Mini proyek menyatukan seluruh konsep Bab 11: training di laptop, ekspor model,
inferensi lokal per sampel, lalu hasil diteruskan ke log CSV dan/atau broker MQTT.
Prediksi yang diterbitkan ke topik `iotlab/room01/prediction` terverifikasi diterima
subscriber. Log simulasi tersimpan di `outputs/reports/edge_room_predictions.csv`.